# 04 — Model: U-Net Segmentation

## Approach

We use a **raster segmentation** approach:
1. Render each BGT tile as a multi-channel image (one channel per class)
2. Render the corresponding BRT tile as the target segmentation mask
3. Train a **U-Net** to predict BRT masks from BGT images
4. At inference: render BGT → U-Net → predicted mask → vectorize back to polygons

### Why U-Net?
- Well-understood, lots of reference implementations
- Works well on spatial data with local patterns
- Skip connections preserve fine-grained geometry details
- Runs on free Colab GPU

### Limitations (documented honestly)
- Rasterization loses exact vertex positions → post-vectorization geometry is approximate
- Resolution caps geometric precision to `tile_size / image_size` metres per pixel
- No explicit topology enforcement between adjacent polygons

Run **01_data_prep.ipynb** and **02_eda.ipynb** first.


## 0. Install dependencies (Colab)

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install torch torchvision geopandas rasterio shapely --quiet

## 1. CONFIG

In [ ]:
from pathlib import Path

CONFIG = {
    "output_root":   Path("processed"),
    "model_dir":     Path("models"),
    "crs":           "EPSG:28992",

    # ── Rasterization ────────────────────────────
    # Pixel resolution of rendered tiles.
    # 256 = 256×256 pixels per tile.
    # Higher = more detail, more memory, slower training.
    # 256 is a good starting point for 500m tiles → ~2m/pixel
    "image_size":    256,

    # ── Model architecture ───────────────────────
    # Number of input channels = number of BGT classes + 1 (background)
    # Set this after running EDA to know how many classes you have.
    # Leave as None to auto-detect from encoder.
    "n_input_channels":  None,   # auto-detected below
    "n_output_classes":  None,   # auto-detected below

    # U-Net feature sizes at each encoder level
    # [32, 64, 128, 256] is lightweight; [64, 128, 256, 512] is heavier
    "unet_features":     [32, 64, 128, 256],

    # ── Training ─────────────────────────────────
    "batch_size":        4,
    "num_epochs":        30,
    "learning_rate":     1e-3,
    "lr_decay_step":     10,     # decay LR every N epochs
    "lr_decay_gamma":    0.5,    # multiply LR by this at each step
    "weight_decay":      1e-4,   # L2 regularisation

    # Stop training if val loss does not improve for this many epochs
    "early_stop_patience": 7,

    # Fraction of training tiles to validate on (if val set is small)
    # Set to None to use full val set from tile_index
    "val_tile_limit":    None,

    # Random seed for reproducibility
    "seed":              42,
}

CONFIG["model_dir"].mkdir(parents=True, exist_ok=True)
print("CONFIG loaded")

## 2. Imports

In [ ]:
import json
import pickle
import random
import warnings
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from shapely.errors import ShapelyDeprecationWarning
warnings.filterwarnings("ignore", category=ShapelyDeprecationWarning)

# Reproducibility
random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 3. Load encoders and auto-detect class counts

In [ ]:
def load_encoder(path):
    """
    Load a LabelEncoder from a JSON file.
    No pickle, no class definition needed — works across all notebooks.
    """
    with open(path) as f:
        data = json.load(f)

    class Encoder:
        pass

    enc = Encoder()
    enc.unknown_label = data["unknown_label"]
    enc.classes_      = data["classes"]
    enc.label_to_int  = data["label_to_int"]
    enc.int_to_label  = {int(k): v for k, v in data["int_to_label"].items()}
    enc._fitted       = True
    return enc


bgt_encoder = load_encoder(CONFIG["output_root"] / "bgt_encoder.json")
brt_encoder = load_encoder(CONFIG["output_root"] / "brt_encoder.json")

# Auto-detect channel counts
CONFIG["n_input_channels"] = len(bgt_encoder.classes_)
CONFIG["n_output_classes"]  = len(brt_encoder.classes_)

print(f"BGT classes (input channels) : {CONFIG['n_input_channels']}")
print(f"BRT classes (output classes) : {CONFIG['n_output_classes']}")

assert CONFIG["n_input_channels"] > 0, "BGT encoder is empty — re-run 01_data_prep.ipynb"
assert CONFIG["n_output_classes"]  > 0, "BRT encoder is empty — re-run 01_data_prep.ipynb"

## 4. Rasterization — convert vector tiles to images

In [ ]:
def rasterize_tile(gdf: gpd.GeoDataFrame,
                   class_col_encoded: str,
                   n_classes: int,
                   image_size: int) -> np.ndarray:
    """
    Render a GeoDataFrame tile to a (n_classes, H, W) float32 array.
    Each channel is a binary mask: 1 where that class exists, 0 elsewhere.
    Returns: numpy array of shape (n_classes, image_size, image_size)
    """
    from rasterio.features import rasterize as rio_rasterize
    from rasterio.transform import from_bounds

    # Guard: n_classes=0 means encoders weren't loaded — fail loud and early
    assert n_classes > 0, (
        f"n_classes is 0 — encoders not loaded correctly. "
        f"Re-run the encoder loading cell above before this one."
    )

    if len(gdf) == 0:
        return np.zeros((n_classes, image_size, image_size), dtype=np.float32)

    minx, miny, maxx, maxy = gdf.total_bounds
    if (maxx - minx) == 0 or (maxy - miny) == 0:
        return np.zeros((n_classes, image_size, image_size), dtype=np.float32)

    transform = from_bounds(minx, miny, maxx, maxy, image_size, image_size)
    canvas = np.zeros((n_classes, image_size, image_size), dtype=np.float32)

    if class_col_encoded not in gdf.columns:
        # Column missing — paint all polygons into channel 0 as fallback
        # This usually means bgt_class_col in CONFIG doesn't match actual column names.
        # Run the diagnostic cell below to find the correct column name.
        shapes = [(geom, 1) for geom in gdf.geometry if geom and not geom.is_empty]
        if shapes:
            canvas[0] = rio_rasterize(
                shapes, out_shape=(image_size, image_size),
                transform=transform, fill=0, dtype=np.float32
            )
        return canvas

    for class_idx in range(n_classes):
        subset = gdf[gdf[class_col_encoded] == class_idx]
        if len(subset) == 0:
            continue
        shapes = [
            (geom, 1)
            for geom in subset.geometry
            if geom is not None and not geom.is_empty
        ]
        if shapes:
            canvas[class_idx] = rio_rasterize(
                shapes, out_shape=(image_size, image_size),
                transform=transform, fill=0, dtype=np.float32
            )

    return canvas


def rasterize_target(gdf: gpd.GeoDataFrame,
                     class_col_encoded: str,
                     n_classes: int,
                     image_size: int) -> np.ndarray:
    """
    Render BRT tile to a (H, W) int64 label map.
    Each pixel gets the class index of the top polygon covering it.
    Background = 0.
    """
    from rasterio.features import rasterize as rio_rasterize
    from rasterio.transform import from_bounds

    if len(gdf) == 0:
        return np.zeros((image_size, image_size), dtype=np.int64)

    minx, miny, maxx, maxy = gdf.total_bounds
    if maxx == minx or maxy == miny:
        return np.zeros((image_size, image_size), dtype=np.int64)

    transform = from_bounds(minx, miny, maxx, maxy, image_size, image_size)

    col = class_col_encoded if class_col_encoded in gdf.columns else None

    if col:
        shapes = [
            (geom, int(cls))
            for geom, cls in zip(gdf.geometry, gdf[col])
            if geom is not None and not geom.is_empty
        ]
    else:
        shapes = [(geom, 1) for geom in gdf.geometry if geom and not geom.is_empty]

    if not shapes:
        return np.zeros((image_size, image_size), dtype=np.int64)

    label_map = rio_rasterize(
        shapes, out_shape=(image_size, image_size),
        transform=transform, fill=0, dtype=np.int32
    )
    return label_map.astype(np.int64)


print("Rasterization functions defined")

## 5. PyTorch Dataset

In [ ]:
class GeneralizationDataset(Dataset):
    """
    Dataset that loads BGT/BRT tile pairs and rasterizes them on the fly.

    __getitem__ returns:
      x : (n_bgt_classes, H, W) float32 tensor  — BGT input image
      y : (H, W) int64 tensor                   — BRT target label map
    """

    def __init__(self, tile_list: list, bgt_encoder, brt_encoder, config: dict):
        self.tiles       = tile_list
        self.bgt_encoder = bgt_encoder
        self.brt_encoder = brt_encoder
        self.config      = config

        # Derive encoded column names from CONFIG so they stay in sync with 01_data_prep.
        # If column names are wrong, the diagnostic cell below will catch it.
        bgt_col = config["bgt_class_col"]   # e.g. 'plus-type' or 'bgt_type'
        brt_col = config["brt_class_col"]   # e.g. 'typeweg' or 'typewegde'
        self.bgt_enc_col = f"{bgt_col}_encoded"
        self.brt_enc_col = f"{brt_col}_encoded"

    def __len__(self):
        return len(self.tiles)

    def __getitem__(self, idx):
        tile = self.tiles[idx]

        bgt = gpd.read_file(tile["bgt"])
        brt = gpd.read_file(tile["brt"])

        # Align BRT extent to BGT extent (important for consistent rasterization)
        brt = brt.cx[
            bgt.total_bounds[0]:bgt.total_bounds[2],
            bgt.total_bounds[1]:bgt.total_bounds[3],
        ]

        img_size = self.config["image_size"]
        n_bgt    = self.config["n_input_channels"]
        n_brt    = self.config["n_output_classes"]

        x = rasterize_tile(bgt, self.bgt_enc_col, n_bgt, img_size)
        y = rasterize_target(brt, self.brt_enc_col, n_brt, img_size)

        return torch.from_numpy(x), torch.from_numpy(y)


# Load tile index
with open(CONFIG["output_root"] / "tile_index.json") as f:
    tile_index = json.load(f)

train_dataset = GeneralizationDataset(
    tile_index["train"], bgt_encoder, brt_encoder, CONFIG
)
val_tiles = tile_index["val"]
if CONFIG["val_tile_limit"]:
    val_tiles = val_tiles[:CONFIG["val_tile_limit"]]

val_dataset = GeneralizationDataset(
    val_tiles, bgt_encoder, brt_encoder, CONFIG
)

print(f"Train tiles : {len(train_dataset)}")
print(f"Val tiles   : {len(val_dataset)}")
print(f"BGT encoded col : '{train_dataset.bgt_enc_col}'")
print(f"BRT encoded col : '{train_dataset.brt_enc_col}'")

## 5b. Diagnostic — verify columns and shapes before training

Run this cell before building the model. It checks that:
- The encoded columns actually exist in the saved tiles
- n_input_channels and n_output_classes are non-zero
- A sample item loads and has the correct tensor shapes

If the encoded column is missing, update `bgt_class_col` / `brt_class_col` in CONFIG
to match the real attribute names printed below.

In [ ]:
# ── Step 1: inspect actual column names in a saved tile ──
sample_tile = tile_index["train"][0]
bgt_check = gpd.read_file(sample_tile["bgt"])
brt_check = gpd.read_file(sample_tile["brt"])

print("BGT tile columns:")
print(" ", bgt_check.columns.tolist())
print("\nBRT tile columns:")
print(" ", brt_check.columns.tolist())

# ── Step 2: check expected encoded columns are present ──
bgt_enc_col = train_dataset.bgt_enc_col
brt_enc_col = train_dataset.brt_enc_col

bgt_ok = bgt_enc_col in bgt_check.columns
brt_ok = brt_enc_col in brt_check.columns

print(f"\nExpected BGT encoded col '{bgt_enc_col}': {'✓ found' if bgt_ok else '✗ MISSING'}")
print(f"Expected BRT encoded col '{brt_enc_col}': {'✓ found' if brt_ok else '✗ MISSING'}")

if not bgt_ok or not brt_ok:
    print("\n⚠ Column mismatch — update bgt_class_col / brt_class_col in CONFIG")
    print("  to match the actual column names printed above, then re-run notebook 01.")
else:
    # ── Step 3: load one sample and check tensor shapes ──
    x_sample, y_sample = train_dataset[0]
    print(f"\nSample x shape : {x_sample.shape}  (expected: [{CONFIG['n_input_channels']}, {CONFIG['image_size']}, {CONFIG['image_size']}])")
    print(f"Sample y shape : {y_sample.shape}  (expected: [{CONFIG['image_size']}, {CONFIG['image_size']}])")
    print(f"x dtype : {x_sample.dtype}  (expected: torch.float32)")
    print(f"y dtype : {y_sample.dtype}  (expected: torch.int64)")
    print(f"y unique classes in sample: {y_sample.unique().tolist()}")
    print("\n✓ Dataset looks good — proceed to model building")

# ── Step 4: build data loaders ──
train_loader = DataLoader(train_dataset, batch_size=CONFIG["batch_size"],
                           shuffle=True,  num_workers=2, pin_memory=(DEVICE=="cuda"))
val_loader   = DataLoader(val_dataset,   batch_size=CONFIG["batch_size"],
                           shuffle=False, num_workers=2, pin_memory=(DEVICE=="cuda"))
print("\nData loaders ready.")

## 6. U-Net architecture

In [ ]:
class ConvBlock(nn.Module):
    """Two consecutive Conv → BN → ReLU layers. The basic U-Net building block."""

    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    """
    U-Net for semantic segmentation.

    in_channels  : number of BGT class channels
    out_channels : number of BRT classes
    features     : list of feature sizes at each encoder level
    """

    def __init__(self, in_channels: int, out_channels: int, features: list):
        super().__init__()

        self.encoders   = nn.ModuleList()
        self.pools      = nn.ModuleList()
        self.decoders   = nn.ModuleList()
        self.upconvs    = nn.ModuleList()

        # ── Encoder path ───────────────────────────
        ch = in_channels
        for feat in features:
            self.encoders.append(ConvBlock(ch, feat))
            self.pools.append(nn.MaxPool2d(2))
            ch = feat

        # ── Bottleneck ─────────────────────────────
        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        # ── Decoder path ───────────────────────────
        rev_features = features[::-1]
        ch = features[-1] * 2
        for feat in rev_features:
            self.upconvs.append(nn.ConvTranspose2d(ch, feat, kernel_size=2, stride=2))
            # After concatenating skip connection: feat (up) + feat (skip) = feat*2
            self.decoders.append(ConvBlock(feat * 2, feat))
            ch = feat

        # ── Output head ────────────────────────────
        self.output_conv = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skip_connections = []

        # Encoder
        for encoder, pool in zip(self.encoders, self.pools):
            x = encoder(x)
            skip_connections.append(x)
            x = pool(x)

        # Bottleneck
        x = self.bottleneck(x)

        # Decoder
        skip_connections = skip_connections[::-1]
        for upconv, decoder, skip in zip(self.upconvs, self.decoders, skip_connections):
            x = upconv(x)
            # Handle size mismatch from odd dimensions
            if x.shape != skip.shape:
                x = nn.functional.interpolate(x, size=skip.shape[2:])
            x = torch.cat([skip, x], dim=1)
            x = decoder(x)

        return self.output_conv(x)


# Build model
model = UNet(
    in_channels  = CONFIG["n_input_channels"],
    out_channels = CONFIG["n_output_classes"],
    features     = CONFIG["unet_features"],
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"U-Net parameters: {total_params:,}")

# Quick forward pass to check shapes
with torch.no_grad():
    dummy = torch.zeros(1, CONFIG["n_input_channels"],
                        CONFIG["image_size"], CONFIG["image_size"]).to(DEVICE)
    out = model(dummy)
    print(f"Output shape: {out.shape}  — expected (1, {CONFIG['n_output_classes']}, "
          f"{CONFIG['image_size']}, {CONFIG['image_size']})")

## 7. Training loop

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    """Train for one epoch. Returns mean loss."""
    model.train()
    total_loss = 0.0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        pred  = model(x)
        loss  = criterion(pred, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


@torch.no_grad()
def validate(model, loader, criterion, device):
    """Validation pass. Returns (mean_loss, mean_pixel_accuracy)."""
    model.eval()
    total_loss = 0.0
    correct    = 0
    total_pix  = 0

    for x, y in loader:
        x = x.to(device)
        y = y.to(device)

        pred  = model(x)
        loss  = criterion(pred, y)
        total_loss += loss.item()

        preds     = pred.argmax(dim=1)
        correct  += (preds == y).sum().item()
        total_pix += y.numel()

    mean_loss = total_loss / len(loader)
    pixel_acc = correct / total_pix
    return mean_loss, pixel_acc


# Loss: CrossEntropy is standard for multi-class segmentation
criterion = nn.CrossEntropyLoss(ignore_index=0)  # ignore background class 0

optimizer = optim.Adam(
    model.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"]
)

scheduler = optim.lr_scheduler.StepLR(
    optimizer,
    step_size=CONFIG["lr_decay_step"],
    gamma=CONFIG["lr_decay_gamma"]
)

print("Training setup ready")
print(f"  Criterion : CrossEntropyLoss (ignores background class 0)")
print(f"  Optimizer : Adam  lr={CONFIG['learning_rate']}  wd={CONFIG['weight_decay']}")
print(f"  Scheduler : StepLR  step={CONFIG['lr_decay_step']}  gamma={CONFIG['lr_decay_gamma']}")

In [ ]:
train_losses = []
val_losses   = []
val_accs     = []

best_val_loss    = float("inf")
epochs_no_improve = 0
best_model_path  = CONFIG["model_dir"] / "best_unet.pt"

print(f"Starting training for {CONFIG['num_epochs']} epochs on {DEVICE}")
print("─" * 65)

for epoch in range(1, CONFIG["num_epochs"] + 1):
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss, val_acc = validate(model, val_loader, criterion, DEVICE)

    scheduler.step()

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    # Early stopping & checkpoint
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        improved = "  ← saved"
    else:
        epochs_no_improve += 1
        improved = ""

    print(
        f"Epoch {epoch:3d}/{CONFIG['num_epochs']} │ "
        f"train {train_loss:.4f} │ "
        f"val {val_loss:.4f} │ "
        f"val_acc {val_acc*100:.1f}%"
        f"{improved}"
    )

    if epochs_no_improve >= CONFIG["early_stop_patience"]:
        print(f"\nEarly stop: no improvement for {CONFIG['early_stop_patience']} epochs.")
        break

print(f"\nBest val loss: {best_val_loss:.4f}")
print(f"Model saved to: {best_model_path}")

## 8. Training curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(train_losses, label="Train loss", color="steelblue")
axes[0].plot(val_losses,   label="Val loss",   color="tomato")
axes[0].set_title("Loss curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("CrossEntropy loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot([a * 100 for a in val_accs], color="seagreen")
axes[1].set_title("Validation pixel accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CONFIG["model_dir"] / "training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("Training curves saved.")

## 9. Visualize predictions on one val tile

In [ ]:
# Load best checkpoint
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

# Grab the first val tile
x_sample, y_sample = val_dataset[0]
x_in = x_sample.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    pred_logits = model(x_in)
    pred_mask   = pred_logits.argmax(dim=1).squeeze(0).cpu().numpy()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Show first BGT channel (background vs everything)
bgt_composite = x_sample.sum(dim=0).numpy()  # sum channels for visibility
axes[0].imshow(bgt_composite, cmap="Blues", interpolation="nearest")
axes[0].set_title("BGT input (channel sum)")
axes[0].set_axis_off()

axes[1].imshow(pred_mask, cmap="tab20", interpolation="nearest",
               vmin=0, vmax=CONFIG["n_output_classes"])
axes[1].set_title("Predicted BRT mask")
axes[1].set_axis_off()

axes[2].imshow(y_sample.numpy(), cmap="tab20", interpolation="nearest",
               vmin=0, vmax=CONFIG["n_output_classes"])
axes[2].set_title("True BRT mask")
axes[2].set_axis_off()

plt.suptitle("Val tile: BGT input → U-Net prediction → true BRT", fontsize=13)
plt.tight_layout()
plt.savefig(CONFIG["model_dir"] / "val_prediction_example.png", dpi=150, bbox_inches="tight")
plt.show()